In [5]:
# Manage API
import os, getpass
os.environ["OPENAI_API_KEY"] = getpass.getpass("🔑 Enter OPENAI_API_KEY: ")

🔑 Enter OPENAI_API_KEY: ··········


In [6]:
# Load Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
DOC_PATH = "/content/drive/MyDrive/AI44PT/data/processed/JRGsamples/1.pdf"
CODEBOOK_PATH = "/content/drive/MyDrive/AI44PT/data/processed/codebook_0912.pdf"


Text Abstract


In [10]:
from pypdf import PdfReader
import re, numpy as np, faiss, json, math
from typing import List, Dict, Any

def extract_pdf_blocks(path: str, chunk_chars=1200, overlap=200):
    reader = PdfReader(path)
    blocks, metas = [], []
    for pi, page in enumerate(reader.pages):
        raw = page.extract_text() or ""
        text = re.sub(r"\s+", " ", raw).strip()
        if not text:
            continue
        # 简单按字符切分
        for start in range(0, len(text), chunk_chars - overlap):
            chunk = text[start:start+chunk_chars]
            blocks.append(chunk)
            metas.append({"page": pi+1, "start": start, "source": path})
    return blocks, metas

def embed_texts(texts: List[str], model="text-embedding-3-small"):
    # 单次最大长度有限，必要时可分批
    resp = client.embeddings.create(model=model, input=texts)
    return np.array([e.embedding for e in resp.data], dtype="float32")

def build_faiss(embs: np.ndarray):
    dim = embs.shape[1]
    index = faiss.IndexFlatIP(dim)  # 余弦相似需先归一化；这里用内积 + 预归一化
    # 归一化
    faiss.normalize_L2(embs)
    index.add(embs)
    return index

def topk_search(index, embs_matrix, query_emb, k=5):
    q = query_emb.astype("float32")
    faiss.normalize_L2(q)
    D, I = index.search(q, k)
    return D, I

# === 3.1 处理 Codebook ===
cb_blocks, cb_metas = extract_pdf_blocks(CODEBOOK_PATH, 1200, 200)
cb_embs = embed_texts(cb_blocks)  # 默认 text-embedding-3-small 经济好用
cb_index = build_faiss(cb_embs)

# === 3.2 处理一篇待测论文（示例：取你上传的第一个非 codebook PDF）===
doc_blocks, doc_metas = extract_pdf_blocks(DOC_PATH, 1200, 200)
doc_embs = embed_texts(doc_blocks)
doc_index = build_faiss(doc_embs)



NameError: name 'client' is not defined